<a href="https://colab.research.google.com/github/cvelac4/Algorithm-and-Leetcode/blob/master/PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Title and Explenation

In [19]:
# Predictions of Suporstore data using Advanced ML PySpark.


# Download Pyspark

In [22]:
!pip install PySpark

# Create a Session

In [24]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('SalesForecasting').getOrCreate()

#Load and Explore the Data


In [25]:
#data
path = '/content/1740463998446_a2368e63c8e87f60.csv'
df = spark.read.csv(path, header=True, inferSchema=True )

#Display the Schema
df.printSchema()

#View the sample data
df.show()


root
 |-- ID: integer (nullable = true)
 |-- Order_id: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship _Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_id: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)
 |-- user_id: double (nullable = true)
 |-- state_id: double (nullable = true)
 |-- order_s: string (nullable = true)

+---+--------------+----------+

#Data Processing



*   Convert Data
*   Handdle Missing
Aggregate on daily level







In [26]:
#convert Order_data to data type
from pyspark.sql.functions import to_date, col,sum, dayofmonth, month, year,lag

df = df.withColumn('Order_Date', col('Order_Date').cast('date') )

df.printSchema()

#Aggregate
daily_sales = df.groupBy('Order_Date').agg(sum('Sales').alias('Daily_Sales'))

daily_sales.show()

root
 |-- ID: integer (nullable = true)
 |-- Order_id: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship _Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_id: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)
 |-- user_id: double (nullable = true)
 |-- state_id: double (nullable = true)
 |-- order_s: string (nullable = true)

+----------+------------------+
|

In [27]:
daily_sales.show(10)

#Sort the date.
daily_sales = daily_sales.sort('Order_Date')
daily_sales.show(10)

+----------+------------------+
|Order_Date|       Daily_Sales|
+----------+------------------+
|2021-08-27|           2070.13|
|2024-09-18|1454.7299999999998|
|2021-06-22|          1975.498|
|2022-03-28|           243.344|
|2022-07-31|          3712.162|
|2023-07-15|             380.2|
|2021-10-11|          1381.164|
|2021-01-27|            426.67|
|2023-11-08| 993.9000000000001|
|2024-08-27|5992.0779999999995|
+----------+------------------+
only showing top 10 rows

+----------+-----------------+
|Order_Date|      Daily_Sales|
+----------+-----------------+
|      NULL|          111.104|
|2021-01-03|             NULL|
|2021-01-04|           288.06|
|2021-01-05|           19.536|
|2021-01-06|           4407.1|
|2021-01-07|87.15799999999999|
|2021-01-09|           40.544|
|2021-01-10|            54.83|
|2021-01-11|             9.94|
|2021-01-13|         3553.795|
+----------+-----------------+
only showing top 10 rows



#Featured Engineering

In [30]:
from pyspark.sql.functions import dayofmonth, month, year, lag
from pyspark.sql.window import Window

#Window
wind_spec = Window.orderBy('Order_Date')

#Add Lag (previous day sales)
daily_sales = daily_sales.withColumn('Prev_Day_Sales', lag('Daily_Sales').over(wind_spec))

daily_sales = daily_sales.withColumn('Day', dayofmonth(col('Order_Date')))
daily_sales = daily_sales.withColumn('Month', month(col('Order_Date')))
daily_sales = daily_sales.withColumn('Year', year(col('Order_Date')))

#Drop NA values
daily_sales = daily_sales.na.drop()

daily_sales.show(10)

+----------+-----------------+-----------------+---+-----+----+
|Order_Date|      Daily_Sales|   Prev_Day_Sales|Day|Month|Year|
+----------+-----------------+-----------------+---+-----+----+
|2021-01-05|           19.536|           288.06|  5|    1|2021|
|2021-01-06|           4407.1|           19.536|  6|    1|2021|
|2021-01-07|87.15799999999999|           4407.1|  7|    1|2021|
|2021-01-09|           40.544|87.15799999999999|  9|    1|2021|
|2021-01-10|            54.83|           40.544| 10|    1|2021|
|2021-01-11|             9.94|            54.83| 11|    1|2021|
|2021-01-13|         3553.795|             9.94| 13|    1|2021|
|2021-01-14|            61.96|         3553.795| 14|    1|2021|
|2021-01-15|           149.95|            61.96| 15|    1|2021|
|2021-01-16|          299.964|           149.95| 16|    1|2021|
+----------+-----------------+-----------------+---+-----+----+
only showing top 10 rows



#Training and Testing

In [31]:
from pyspark.ml.feature import VectorAssembler

#Assemble the fetures into singal vector
feature_col = ['Day', 'Month', 'Year', 'Prev_Day_Sales']
assembler = VectorAssembler(inputCols = feature_col, outputCol = 'features')

daily_sales_update = assembler.transform(daily_sales).select('features', 'Daily_Sales')

daily_sales_update.show(10)

+--------------------+-----------------+
|            features|      Daily_Sales|
+--------------------+-----------------+
|[5.0,1.0,2021.0,2...|           19.536|
|[6.0,1.0,2021.0,1...|           4407.1|
|[7.0,1.0,2021.0,4...|87.15799999999999|
|[9.0,1.0,2021.0,8...|           40.544|
|[10.0,1.0,2021.0,...|            54.83|
|[11.0,1.0,2021.0,...|             9.94|
|[13.0,1.0,2021.0,...|         3553.795|
|[14.0,1.0,2021.0,...|            61.96|
|[15.0,1.0,2021.0,...|           149.95|
|[16.0,1.0,2021.0,...|          299.964|
+--------------------+-----------------+
only showing top 10 rows



In [32]:
train, test = daily_sales_update.randomSplit([0.8, 0.2], seed= 42)